# Decision Tree Models — Random Forest & XGBoost

**MSE 546 — AQI Forecasting (Group 2)**  
This notebook implements **Random Forest** and **XGBoost** for the same regression task as the baseline: predict **daily AQI** given city, pollutants, calendar, and historical AQI.

**Why tree models here:**
- **Task:** Regression — predict a continuous AQI value (same target as Ridge).
- **Nonlinearity:** Trees capture interactions and thresholds (e.g. "if PM2.5 > X and season == winter then higher AQI") that Ridge cannot.
- **Severe days:** Ridge over-smooths extremes (MAE on Severe ≈ 201); forests and boosting often do better on tail events.
- **Same pipeline:** We use the same features, temporal split, log-target training, and evaluation (RMSE, MAE, R², per-category MAE) as in `preprocessing.py` and the baseline.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

from preprocessing import (
    load_and_prepare, impute, build_features,
    get_train_test, get_arrays, evaluate, save_results,
    AQI_LABELS,
)

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## 1. Load data and get train/test arrays

Same as baseline: temporal split at 2019-12-01, features from `build_features` (pollutants, missing flags, calendar, AQI lags/rolling, city dummies). Target is **log1p(AQI)** for training; we evaluate on the **original AQI scale** after applying `expm1` to predictions.

In [ ]:
df = load_and_prepare()
df = impute(df)
df, feature_cols = build_features(df)
train_df, test_df = get_train_test(df)
X_train, X_test, y_train_log, y_test_raw, y_test_log = get_arrays(train_df, test_df, feature_cols)

print(f'Features: {len(feature_cols)}')
print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')
print('Target: train on log1p(AQI), evaluate on raw AQI (expm1(pred))')

## 2. Random Forest

No scaling needed for tree-based models. We train on `y_train_log` and convert predictions back to AQI with `np.expm1`, clipped to non-negative.

In [ ]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=10,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train_log)

y_pred_log_rf = rf.predict(X_test)
y_pred_raw_rf = np.clip(np.expm1(y_pred_log_rf), 0, None)

result_rf = evaluate(y_test_raw, y_pred_raw_rf, label='Random Forest')
save_results(result_rf, 'random_forest')

## 3. XGBoost

XGBoost fits additive trees by gradient boosting. Same target: `y_train_log`. We invert with `expm1` for evaluation.

In [ ]:
xgb_model = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    objective='reg:squarederror',
)
xgb_model.fit(X_train, y_train_log)

y_pred_log_xgb = xgb_model.predict(X_test)
y_pred_raw_xgb = np.clip(np.expm1(y_pred_log_xgb), 0, None)

result_xgb = evaluate(y_test_raw, y_pred_raw_xgb, label='XGBoost')
save_results(result_xgb, 'xgboost')

## 4. Feature importance (optional)

Tree models provide feature importances; useful for reporting which drivers (e.g. PM2.5, lags, season) matter most.

In [ ]:
def plot_importance(importances, names, top_n=20, title='Feature importance'):
    order = np.argsort(importances)[-top_n:]
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh(range(top_n), importances[order])
    ax.set_yticks(range(top_n))
    ax.set_yticklabels([names[i] for i in order], fontsize=9)
    ax.set_xlabel('Importance')
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

plot_importance(rf.feature_importances_, feature_cols, title='Random Forest — Top 20')
plot_importance(xgb_model.feature_importances_, feature_cols, title='XGBoost — Top 20')

## 5. (Optional) Hyperparameter tuning

Uncomment and run to search over a small grid. Use temporal split or a single validation fold to avoid leakage; here we use 3-fold CV for speed (same temporal data in each fold). For stricter temporal evaluation, use a fixed validation set (e.g. last 20% of train dates).

In [ ]:
# Example: RandomizedSearchCV for Random Forest (optional)
# param_dist = {
#     'n_estimators': [100, 200, 300],
#     'max_depth': [10, 15, 20],
#     'min_samples_leaf': [5, 10, 20],
#     'max_features': ['sqrt', 0.5],
# }
# search = RandomizedSearchCV(
#     RandomForestRegressor(random_state=42, n_jobs=-1),
#     param_distributions=param_dist, n_iter=15, cv=3, scoring='neg_mean_squared_error',
#     random_state=42, n_jobs=-1,
# )
# search.fit(X_train, y_train_log)
# best_rf = search.best_estimator_
# y_pred_raw_tuned = np.clip(np.expm1(best_rf.predict(X_test)), 0, None)
# evaluate(y_test_raw, y_pred_raw_tuned, label='Random Forest (tuned)')
pass